# PointVisor – Point-Supervised Semantic Segmentation on DLRSD
**LandVisor Project Task Solution**

Implements partial Focal CE loss, simulates point annotations on real remote sensing data, trains a segmentation model, and runs experiments.

In [ ]:
# === DEBUG CELL ===
print("DATA_ROOT  :", DATA_ROOT)
print("IMAGE_DIR  :", IMAGE_DIR)
print("LABEL_DIR  :", LABEL_DIR)

all_jpg = glob(os.path.join(IMAGE_DIR, "**/*.jpg"), recursive=True)
all_png = glob(os.path.join(IMAGE_DIR, "**/*.png"), recursive=True)

print(f"Found {len(all_jpg)} .jpg files")
print(f"Found {len(all_png)} .png files")
print("First 5 image paths:")
for p in (all_jpg + all_png)[:5]:
    print("   ", p)

In [ ]:
class PartialFocalLoss(nn.Module):
    """Partial Focal CE Loss"""
    def __init__(self, gamma=2.0, alpha=0.25, ignore_index=255):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, pred, target, mask):
        # pred: (B, C, H, W) logits
        # target: (B, H, W) long
        # mask: (B, H, W) float 0/1 (only labeled points = 1)
        ce = nn.functional.cross_entropy(pred, target, reduction='none', ignore_index=self.ignore_index)
        pt = torch.exp(-ce)
        focal = self.alpha * (1 - pt) ** self.gamma * ce
        masked = focal * mask
        return masked.sum() / (mask.sum() + 1e-8)   # average only over labeled points

In [ ]:
class DLRSDPointDataset(Dataset):
    def __init__(self, image_files, points_per_class=5):
        self.image_files = image_files
        self.points_per_class = points_per_class

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
    
        # Safer label path (works with subfolders)
        rel_path = os.path.relpath(img_path, IMAGE_DIR)
        label_path = os.path.join(LABEL_DIR, rel_path).replace('.jpg', '.png').replace('.JPG', '.png')

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))

        label = cv2.imread(label_path, cv2.IMREAD_GRAYSCALE)  # 0-16 classes
        label = label.astype(np.int64)

        # Simulate point labels
        point_target, point_mask = self.simulate_points(label)

        return torch.from_numpy(img), torch.from_numpy(label), point_target, point_mask

    def simulate_points(self, label_np):
        H, W = label_np.shape
        point_target = np.full((H, W), 255, dtype=np.int64)
        point_mask = np.zeros((H, W), dtype=np.float32)

        for c in range(17):  # 17 classes in DLRSD
            ys, xs = np.where(label_np == c)
            if len(ys) == 0: continue
            n = min(self.points_per_class, len(ys))
            idx = random.sample(range(len(ys)), n)
            point_target[ys[idx], xs[idx]] = c
            point_mask[ys[idx], xs[idx]] = 1.0

        return point_target, point_mask

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = smp.Unet(encoder_name='resnet34', encoder_weights='imagenet',
                 in_channels=3, classes=17).to(device)

criterion = PartialFocalLoss(gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

def train_one_epoch(dataloader, points_per_class):
    model.train()
    total_loss = 0
    for img, full_label, point_target, point_mask in tqdm(dataloader):
        img, point_target, point_mask = img.to(device), point_target.to(device), point_mask.to(device)
        
        optimizer.zero_grad()
        pred = model(img)
        loss = criterion(pred, point_target, point_mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(dataloader)

Run the cell below to see an example image + points